# Impact Analyzer

This notebook demonstrates the **Impact Analyzer** module in pdstools, including:

- Loading data from PDC (monitoring export) or VBD (scenario planner) sources
- Summarizing experiments and control groups
- Computing **engagement lift** and **value lift** with confidence intervals
- Using the new **statistics module** for transparent, auditable calculations
- **Sample size planning** for new experiments
- Built-in plotting (overview and trend charts)

> **📦 Optional dependencies**
>
> This article uses features from the pdstools `adm` extra. Install with your favorite package manager, e.g. `uv pip install "pdstools[adm]"`.

## 1. Setup

In [ ]:
from pdstools import ImpactAnalyzer
from pdstools.impactanalyzer.statistics import (
    FORMULAS,
    LiftResult,
    Z_95,
    accept_rate,
    binomial_se,
    calculate_engagement_lift,
    calculate_value_lift,
    is_significant,
    lift_se,
    required_sample_size,
)

## 2. Loading data

VBD Scenario Planner Actuals exports (zip files) are the preferred input format.

The sample multi-day VBD data ships with the repository.

In [ ]:
import tempfile
import urllib.request
from pathlib import Path

# Load sample VBD data. Prefer the local copy if running from a repo
# checkout (works on PR branches where the file isn't on master yet);
# otherwise download it from GitHub.
sample_vbd_filename = "ImpactAnalyzer_InfinityDemo.zip"
for candidate in [Path.cwd(), *Path.cwd().parents]:
    bundled = candidate / "data" / "ia" / sample_vbd_filename
    if bundled.exists():
        sample_vbd_path = bundled
        break
else:
    sample_vbd_url = (
        "https://github.com/pegasystems/pega-datascientist-tools/"
        f"raw/master/data/ia/{sample_vbd_filename}"
    )
    with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as tmp:
        with urllib.request.urlopen(sample_vbd_url) as response:
            tmp.write(response.read())
        sample_vbd_path = Path(tmp.name)

ia = ImpactAnalyzer.from_vbd(str(sample_vbd_path))
ia.ia_data.head(10).collect()


The data is normalised into a long format with Impressions, Accepts, Channel, ControlGroup, etc.

In [ ]:
print(f"Loaded {ia.ia_data.select('SnapshotTime').unique().collect().height} snapshots")

## 3. Summarizing experiments

In [ ]:
ia.summarize_control_groups().collect()

In [ ]:
ia.summarize_experiments().collect()

In [ ]:
ia.summarize_experiments(by="Channel").collect()

In [ ]:
ia.overall_summary().collect()

In [ ]:
ia.summary_by_channel().collect()

## 4. Statistics module — transparent lift calculations

The statistics module exposes every formula used by Impact Analyzer as plain Python functions.
Each has a corresponding Formula object in the FORMULAS registry with LaTeX and description.

### 4a. Formula registry

In [ ]:
for key, f in FORMULAS.items():
    print(f"{key:20s}  {f.description[:80]}")

### 4b. Computing engagement lift step by step

Suppose an experiment has:
- **Test**: 50,000 impressions, 3,500 accepts
- **Control**: 5,000 impressions, 300 accepts

In [ ]:
n_test, a_test = 50_000, 3_500
n_ctrl, a_ctrl = 5_000, 300

# Step 1: Accept rates
p_test = accept_rate(a_test, n_test)
p_ctrl = accept_rate(a_ctrl, n_ctrl)
print(f"Accept rate  test={p_test:.6f}  control={p_ctrl:.6f}")

# Step 2: Standard errors
se_test = binomial_se(a_test, n_test)
se_ctrl = binomial_se(a_ctrl, n_ctrl)
print(f"SE           test={se_test:.8f}  control={se_ctrl:.8f}")

# Step 3: Engagement lift with CI
result: LiftResult = calculate_engagement_lift(a_test, n_test, a_ctrl, n_ctrl)
print(f"Engagement Lift = {result.lift:.6f} ({result.lift*100:.4f}%)")
print(f"SE(Lift)        = {result.se:.8f}")
print(f"95% CI          = [{result.lift - result.ci_95():.6f}, {result.lift + result.ci_95():.6f}]")
print(f"Significant?    = {result.significant}")

### 4c. The LiftResult object

All lift functions return a frozen LiftResult dataclass.

In [ ]:
print(result)
print(f"Half-width of 95% CI: {result.ci_95():.8f}")

### 4d. Significance testing

In [ ]:
# A small experiment that is NOT significant
small = calculate_engagement_lift(35, 500, 30, 500)
print(f"Small experiment: lift={small.lift:.4f}, significant={small.significant}")

# The larger experiment from above IS significant
print(f"Large experiment: lift={result.lift:.4f}, significant={result.significant}")

## 5. Sample size planning

Estimate how many impressions you need to detect a given lift.

In [ ]:
n = required_sample_size(baseline_rate=0.06, mde=0.05, control_ratio=0.10)
print(f"Required total impressions: {n:,}")
print(f"  Control arm: {int(n * 0.10):,}")
print(f"  Test arm:    {int(n * 0.90):,}")

## 6. Plotting

In [ ]:
ia.plot.overview(facet="Channel")

In [ ]:
ia.plot.trend(facet="Channel", every="1w")

## 7. Interactive Streamlit app

For a full interactive experience with forest plots, drill-down analysis,
formula walkthroughs, and live data generation, launch the Streamlit app:

```bash
pdstools run impact_analyzer
```

The app includes:
- **Overall Summary** with CI bands, EWMA smoothing, and forest plots
- **Drill Down** per-channel lift analysis with cannibalization warnings
- **About** methodology documentation